##Imports

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

##Parameters

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.silver_product"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.gold_dim_product"

##Read Silver Product

In [0]:
product_df = spark.table(SOURCE_TABLE)

##Build Source Dataset

In [0]:
source_df = (
    product_df
    .select(
        F.col("ProductID"),

        F.col("Name").alias("ProductName"),

        F.col("Color"),

        F.col("StandardCost")
            .cast("decimal(18,2)")
            .alias("StandardCost"),

        F.col("ListPrice")
            .cast("decimal(18,2)")
            .alias("ListPrice")
    )
)

##Create Product Hash

In [0]:
source_df = source_df.withColumn(
    "product_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("ProductID").cast("string"),
            F.col("ProductName"),
            F.col("Color"),
            F.col("StandardCost").cast("string"),
            F.col("ListPrice").cast("string")
        ),
        256
    )
)

##Add SCD Columns

In [0]:
source_df = (
    source_df
    .withColumn(
        "effective_start_date",
        F.current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        F.to_timestamp(
            F.lit("9999-12-31 23:59:59")
        )
    )
    .withColumn(
        "is_current",
        F.lit(1)
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

##Create Gold Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.final.gold_dim_product
(
    Product_SK BIGINT GENERATED ALWAYS AS IDENTITY,

    ProductID BIGINT,
    ProductName STRING,
    Color STRING,

    StandardCost DECIMAL(18,2),
    ListPrice DECIMAL(18,2),

    product_hash STRING,

    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,

    is_current INT,

    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA

##Initial Load Check

In [0]:
target_count = spark.table(TARGET_TABLE).count()

##Initial Load

In [0]:
if target_count == 0:

    source_df.select(
        "ProductID",
        "ProductName",
        "Color",
        "StandardCost",
        "ListPrice",
        "product_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "created_at",
        "updated_at"
    ).write \
     .format("delta") \
     .mode("append") \
     .saveAsTable(TARGET_TABLE)

    print("Initial Product Dimension Load Complete")

##SCD2 Logic

####Load current version:

In [0]:
target_df = (
    spark.table(TARGET_TABLE)
    .filter("is_current = 1")
)

####Detect changes:

In [0]:
changes_df = (
    source_df.alias("s")
    .join(
        target_df.alias("t"),
        "ProductID"
    )
    .filter(
        F.col("s.product_hash")
        !=
        F.col("t.product_hash")
    )
    .select("s.*")
)

####Expire old version:

In [0]:
delta_target = DeltaTable.forName(
    spark,
    TARGET_TABLE
)

In [0]:
(
    delta_target.alias("t")
    .merge(
        changes_df.alias("s"),
        """
        t.ProductID=s.ProductID
        AND t.is_current=1
        """
    )
    .whenMatchedUpdate(
        set={
            "effective_end_date":
                "current_timestamp()",

            "is_current":
                "0",

            "updated_at":
                "current_timestamp()"
        }
    )
    .execute()
)

####Insert new version:

In [0]:
new_versions_df = (
    changes_df
    .withColumn(
        "effective_start_date",
        F.current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        F.to_timestamp(
            F.lit("9999-12-31 23:59:59")
        )
    )
    .withColumn(
        "is_current",
        F.lit(1)
    )
)

In [0]:
new_versions_df = new_versions_df.select(
    "ProductID",
    "ProductName",
    "Color",
    "StandardCost",
    "ListPrice",
    "product_hash",
    "effective_start_date",
    "effective_end_date",
    "is_current",
    "created_at",
    "updated_at"
)

In [0]:
new_versions_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TARGET_TABLE)

##New Products

In [0]:
target_current = (
    spark.table(TARGET_TABLE)
    .filter("is_current = 1")
)

In [0]:
new_product_df = (
    source_df.alias("s")
    .join(
        target_current.alias("t"),
        "ProductID",
        "left_anti"
    )
)

In [0]:
new_product_df = new_product_df.select(
    "ProductID",
    "ProductName",
    "Color",
    "StandardCost",
    "ListPrice",
    "product_hash",
    "effective_start_date",
    "effective_end_date",
    "is_current",
    "created_at",
    "updated_at"
)

In [0]:
new_product_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TARGET_TABLE)

##Optimize

In [0]:
%sql
OPTIMIZE workspace.final.gold_dim_product

##Validation

####Current products:

In [0]:
%sql
SELECT *
FROM workspace.final.gold_dim_product
WHERE is_current = 1

####History:

In [0]:
%sql
SELECT *
FROM workspace.final.gold_dim_product
ORDER BY ProductID,
         effective_start_date